In [ ]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 115.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [ ]:
import os, cv2, yaml, shutil
from roboflow import Roboflow

rf = Roboflow(api_key="xU9P4ByCPu9frBO3w0hH")
project = rf.workspace("sigara-hpnoo").project("truck-over-dimension-brupr")
dataset = project.version(2).download("yolov8")

yolo_dir = dataset.location
with open(os.path.join(yolo_dir, "data.yaml"), "r") as f:
    classes = yaml.safe_load(f).get('names', [])

target_dir = "/content/mobilenet_dataset"
if os.path.exists(target_dir): shutil.rmtree(target_dir)

class_folders = [c.replace(" ", "_").upper() for c in classes]
for split in ['train', 'valid', 'test']:
    for c in class_folders:
        os.makedirs(os.path.join(target_dir, split if split != 'valid' else 'val', c), exist_ok=True)
        img_dir, lbl_dir = os.path.join(yolo_dir, split, "images"), os.path.join(yolo_dir, split, "labels")
        if not os.path.exists(img_dir): continue
        for img_name in os.listdir(img_dir):
            if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')): continue
            base_name = os.path.splitext(img_name)[0]
            lbl_path = os.path.join(lbl_dir, base_name + ".txt")
            if not os.path.exists(lbl_path): continue
            img = cv2.imread(os.path.join(img_dir, img_name))
            if img is None: continue
            h, w = img.shape[:2]
            with open(lbl_path, "r") as f: lines = f.readlines()
            for idx, line in enumerate(lines):
                parts = line.strip().split()
                if len(parts) >= 5 and int(parts[0]) < len(class_folders):

                    # THE FIX IS HERE: Check for Polygon vs BBox
                    if len(parts) > 5:
                        xs = [float(parts[i]) for i in range(1, len(parts), 2)]
                        ys = [float(parts[i]) for i in range(2, len(parts), 2)]
                        x1, y1 = int(min(xs) * w), int(min(ys) * h)
                        x2, y2 = int(max(xs) * w), int(max(ys) * h)
                    else:
                        cx, cy, bw, bh = float(parts[1])*w, float(parts[2])*h, float(parts[3])*w, float(parts[4])*h
                        x1, y1 = max(0, int(cx - bw/2)), max(0, int(cy - bh/2))
                        x2, y2 = min(w, int(cx + bw/2)), min(h, int(cy + bh/2))

                    crop = img[y1:y2, x1:x2]
                    if crop.shape[0] > 0 and crop.shape[1] > 0:
                        cv2.imwrite(os.path.join(target_dir, 'val' if split == 'valid' else split, class_folders[int(parts[0])], f"{base_name}_{idx}.jpg"), crop)

print("\nColab Dataset Cropped and Ready!")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Truck-Over-Dimension-2 in yolov8:: 100%|██████████| 5718/5718 [00:01<00:00, 4612.94it/s]



Colab Dataset Cropped and Ready!


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import os

target_dir = "/content/mobilenet_dataset"

# 1. UPGRADE: Heavy Data Augmentation (Lighting, Zoom, Rotation)
train_generator = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.7, 1.3]
).flow_from_directory(os.path.join(target_dir, "train"), target_size=(224, 224), batch_size=32, class_mode='binary')

val_generator = ImageDataGenerator(rescale=1./255)\
    .flow_from_directory(os.path.join(target_dir, "val"), target_size=(224, 224), batch_size=32, class_mode='binary')

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 2. UPGRADE: Fine-Tuning (Unfreeze the top 30 layers)
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

x = Dropout(0.3)(GlobalAveragePooling2D()(base_model.output)) # Increased dropout slightly for safety
model = Model(inputs=base_model.input, outputs=Dense(1, activation='sigmoid')(x))

# 3. UPGRADE: Super slow learning rate (0.0001) so we don't break the unfreezed layers
model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# 4. UPGRADE: Early Stopping Safety Net
early_stop = EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True)

print("\nTraining Ultimate MobileNet (Fine-Tuned)...")
model.fit(train_generator, epochs=200, validation_data=val_generator, callbacks=[early_stop])

model.save("/content/mobilenet_odol_classifierV2.h5")
print("\nFINISHED! Download mobilenet_odol_classifier.h5 from the Colab file explorer.")


Found 2127 images belonging to 2 classes.
Found 136 images belonging to 2 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Training Ultimate MobileNet (Fine-Tuned)...
Epoch 1/200
67/67 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - accuracy: 0.7687 - loss: 0.4743 - val_accuracy: 0.5515 - val_loss: 0.7177
Epoch 2/200
67/67 ━━━━━━━━━━━━━━━━━━━━ 34s 512ms/step - accuracy: 0.8811 - loss: 0.2853 - val_accuracy: 0.8382 - val_loss: 0.3898
Epoch 3/200
67/67 ━━━━━━━━━━━━━━━━━━━━ 35s 522ms/step - accuracy: 0.9135 - loss: 0.2145 - val_accuracy: 0.7794 - val_loss: 0.5222
Epoch 4/200
67/67 ━━━━━━━━━━━━━━━━━━━━ 35s 528ms/step - accuracy: 0.9351 - loss: 0.1765 - val_accuracy: 0.7868 - val_loss: 0.5173
Epoch 5/200
67/67 ━━━━━━━━━━━━━━━━━━━━ 36s 532ms/step - accuracy: 0.9445 - loss: 0.1443 - val_accuracy: 0.8456 - val_loss: 0.3902
Epoch 6/200
67/67 ━━━━━━━━━━━━━━━━━━━━ 35s 515ms/step - accuracy: 0.9426 - loss: 0.1418 - val_accuracy: 0.8897 - val_loss: 0.3030
Epoch 7/200
67/67 ━━━━━━━━━━━━━━━━━━━━ 36s 53


FINISHED! Download mobilenet_odol_classifier.h5 from the Colab file explorer.
